In [20]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "manrique2013repeated")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Innovation_last_criterion_04.06.10__2_sheet1.csv")
complete_path_2 = os.path.join(original_data_pathway, "Innovation_last_criterion_04.06.10__2_sheet2.csv")
complete_path_3 = os.path.join(original_data_pathway, "Innovation_last_criterion_04.06.10__2_sheet3.csv")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [21]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df1[['day', 'month','year']] = df1['Date'].str.split('/',expand=True)

df2 = pd.read_csv(complete_path_2)
df2[['day','month', 'year']] = df2['Date'].str.split('/',expand=True)

df3 = pd.read_csv(complete_path_3)
df3[['month','day', 'year']] = df3['Date'].str.split('/',expand=True)


In [22]:

data_frames = [df1, df2, df3]

for index, x in enumerate(data_frames):
    x['study_id']="manrique2013repeated"
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s)
    x.rename(columns={"subject": "participant"}, inplace=True)
    x['participant'] = x['participant'].str.rstrip()
    data_frames[index]=x
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [23]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')


In [24]:

fulldf['year'] = fulldf['year'].astype(int)
fulldf['year'].replace(10, 2010, inplace=True)
fulldf['year'].replace(9, 2009, inplace=True)

fulldf.columns = fulldf.columns.str.replace(' ', '_', regex=True)

fulldf['success_action'].replace(' ', '_', inplace=True, regex=True)
# fulldf.columns
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365


In [25]:
fulldf['success_action'].unique()
fulldf.rename(columns={"success_action": "success_action_temp"}, inplace=True)

spe_2=[]  
for index, row in fulldf.iterrows():
    if row['success_action_temp']=='no_succes':
        spe_2.append("no_success")
    else:
        spe_2.append(row['success_action_temp'])
fulldf = fulldf.assign(success_action=spe_2)

In [26]:
studyID_standardized=fulldf[['study_id', 'year','month','day', 'participant', 
                             'age_in_years','sex', 'species',  'session', 'trial', 'apparatus','start_time',
       'end_time', 'finger', 'lift', 'shoot', 'success_action']]
# removed 'level_attained', 'session_attained', 'total_time_to_level', 'success_action_temp'
comp_out_path_stand = os.path.join(out_pathway, 'manrique2013repeated_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'manrique2013repeated_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)